# Pacotes

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time
from keras.models import Sequential
from keras.layers import Dense, LSTM, Bidirectional
from keras.layers import  Masking
from scipy.interpolate import UnivariateSpline,CubicSpline

import seaborn as sns
from functions import  SP_Learner, interpolate
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib
from matplotlib import rc
import warnings
from pathlib import Path

# Select which models to run
Choice are ['SPATIAL', 'CNN', 'LSTM'] and default is that all models are run

In [ ]:
model_select =  ['SPATIAL']

# Select which dataset to run for [ADCP, Temperature, or Dissolved oxygen]

In [ ]:
['ADCP_sensor_', 'Oxygen_Sensor', 'Temp_Sensor']
ts_stem = 'ADCP_sensor_' 

In [ ]:
train_split, test_split = .8 ,1 # We allocate 80% for training and remaining 20% for test
root = '.'
result = root + '/Results/'
error_folder = './errors/'
run_metrics = './running_data/'
mod_summary = './model_summary/'


Path(result).mkdir(parents=True, exist_ok=True)
Path(error_folder).mkdir(parents=True, exist_ok=True)
Path(run_metrics).mkdir(parents=True, exist_ok=True)
Path(mod_summary).mkdir(parents=True, exist_ok=True)


if 'ADCP' in ts_stem:
    sensor_idx = [2,4]
else:
    sensor_idx = [1,8,9]

#load Temperature data from the local file
# The data is resampled with a 30 minutes range
# pick 80%-100% to test
Temp = []
data = pd.DataFrame()
file_name = '../sensor_data/interpolated_data/' + ts_stem
for i in sensor_idx: # We have data quality issues with the other sensors
    df = pd.read_csv(file_name+str(i)+'.csv',index_col="observed_timestamp", parse_dates=True)
    data = data.append(df)
    s = df.size
    print('data_length: ', s)
    df_train = df[:int(train_split*s)]
    df_train[np.isnan(df_train)]=-1.0
    df_test = df[int(train_split*s):int(test_split*s)]
    print(df_train.size, df_test.size)
    Temp.append(df.value.to_list()) # Creating a list of all sensor dataframes

In [ ]:
nlag = 48
Temp_time = df.index[nlag:] # Why? 
test_Temp_time = Temp_time[int(train_split*(len(Temp_time))):]

print([len(Temp[i]) for i in range(len(Temp))])
l = min([len(Temp[i]) for i in range(len(Temp))])
# We want all sensors to begin at same time
Temp = [T[:l] for T in Temp]
print([len(Temp[i]) for i in range(len(Temp))])

if 'ADCP' in ts_stem:
    df = pd.DataFrame({'S1': Temp[0],
                'S2': Temp[1]})
else:
    df = pd.DataFrame({'S1': Temp[0],
            'S2': Temp[1],
            'S3': Temp[2]})
Temp_diff = df.diff(periods=nlag)[nlag:] #diferença em relação a linha anterior
#dataset
Temp_diff = np.transpose(Temp_diff.to_numpy())

In [16]:
import pandas as pd
import numpy as np

In [7]:
df = pd.DataFrame({'S1': [1,2,3],
                'S2': [4,5,6]})

In [20]:
df

,S1,S2
0,1,4
1,2,5
2,3,6


In [19]:
df.diff(periods=2)

,S1,S2
0,NaN,NaN
1,NaN,NaN
2,2.0,2.0


In [17]:
np.transpose(df.diff(periods=2)[2:].to_numpy())

array([[2.],
       [2.]])

In [11]:
dictionary = {"raj": 2, "striver": 3, "vikram": 4}
print(dictionary.values())

dict_values([2, 3, 4])


# Model parameters

In [ ]:
#training info
train_time = 6
predict_time = 1
predict_position = 47
Stride = 1
epoches = 100
batch_size = 512

In [ ]:
#SPATIAL -------------------------- model training 
if 'SPATIAL' in model_select:
    name = '_Temp_diff_' + 'SPATIAL'
    tp_py, tp_ty, tp_error, tp_std, tp_model = SP_Learner(Temp_diff, name,  train_split, test_split,  train_time, predict_time, 
                                                            predict_position, Stride, 0, epoches, run_model='SPATIAL',
                                                            batch_size=batch_size, plot=False)
    print('Temp SPATIAL MEAN: ', data.mean(), 'STD: ',data.std(), 'Skew: ', data.skew())

    #record the errors
    error_file = error_folder + name + '_error.txt'
    open(error_file, 'w').close()
    MAE = []
    MAPE = []
    STD = []
    with open(error_file, "a") as text_file:
        for i in range(len(tp_py)):
            mae = mean_absolute_error(tp_ty[i], tp_py[i])
            MAE.append(mae)
            mape = mean_absolute_percentage_error(tp_ty[i], tp_py[i])
            MAPE.append(mape)
            std = np.std(abs(tp_ty[i] - tp_py[i]))
            STD.append(std)
            info = 'Oxygen sensor {}:   mae: {:.4f},  std: {:.4f}, mape: {:.4f}'.format(i, mae, std, mape)
            text_file.write(info + "\n")
            print('mae: ', mae, 'std: ', std, 'mape: ', mape)
        info = 'Oxygen:   MAE: {:.4f},  STD: {:.4f}, MAPE: {:.4f}'.format(np.mean(MAE), np.mean(STD), np.mean(MAPE))
        text_file.write(info + "\n")

    #save predicted data
    test_time = test_Temp_time[47+6:]
    print(len(test_time), len(tp_ty[0]))
    for i in range(len(tp_py)):
        df = pd.DataFrame({'date': test_time,
                    'value': tp_py[i]})
    #    df.to_csv(root+'/prediction/' + ts_stem +str(sensor_idx[i])+'_SPATIAL_prediction.csv', index=False)

    ytest_spatial = tp_ty
    ypred_spatial = tp_py

In [ ]:
#CNN -------------------------- model training 
if 'CNN' in model_select:
    name = '_Temp_diff_' + 'CNN'
    tp_py, tp_ty, tp_error, tp_std, tp_model = SP_Learner(Temp_diff, name, train_split, test_split, train_time, predict_time, 
                                                        predict_position, Stride, 0, epoches, run_model='CNN',
                                                        batch_size=batch_size,  plot=False)

    print('Temp CNN MEAN: ', data.mean(), 'STD: ',data.std(), 'Skew: ', data.skew())

    #record the errors
    error_file = error_folder + name + '_error.txt'
    open(error_file, 'w').close()
    MAE = []
    MAPE = []
    STD = []
    with open(error_file, "a") as text_file:
        for i in range(len(tp_py)):
            mae = mean_absolute_error(tp_ty[i], tp_py[i])
            MAE.append(mae)
            mape = mean_absolute_percentage_error(tp_ty[i], tp_py[i])
            MAPE.append(mape)
            std = np.std(abs(tp_py[i] - tp_ty[i]))
            STD.append(std)
            info = 'Oxygen sensor {}:   mae: {:.4f},  std: {:.4f}, mape: {:.4f}'.format(i, mae, std, mape)
            text_file.write(info + "\n")
            print('mae: ', mae, 'std: ', std, 'mape: ', mape)
        info = 'Oxygen:   MAE: {:.4f},  STD: {:.4f}, MAPE: {:.4f}'.format(np.mean(MAE), np.mean(STD), np.mean(MAPE))
        text_file.write(info + "\n")

    #save predicted data
    test_time = test_Temp_time[47+6:]
    print(len(test_time), len(tp_ty[0]))

    for i in range(len(tp_py)):
        df = pd.DataFrame({'date': test_time,
                    'value': tp_py[i]})
        df.to_csv(root+'/prediction/' + ts_stem +str(sensor_idx[i])+'_CNN_prediction.csv', index=False)
    ytest_CNN = tp_ty
    ypred_CNN = tp_py

In [ ]:
#LSTM -------------------------- model training 
if 'LSTM' in model_select:
    name = '_Temp_diff_' + 'CNN'
    tp_py, tp_ty, tp_error, tp_std, tp_model = SP_Learner(Temp_diff, name, train_split, test_split, train_time, predict_time, 
                                                        predict_position, Stride, 0, epoches, run_model='LSTM',
                                                        batch_size=batch_size,  plot=False)

    print('Temp LSTM MEAN: ', data.mean(), 'STD: ',data.std(), 'Skew: ', data.skew())

    #record the errors
    error_file = error_folder + name + '_error.txt'
    open(error_file, 'w').close()
    MAE = []
    MAPE = []
    STD = []
    with open(error_file, "a") as text_file:
        for i in range(len(tp_py)):
            mae = mean_absolute_error(tp_ty[i], tp_py[i])
            MAE.append(mae)
            mape = mean_absolute_percentage_error(tp_ty[i], tp_py[i])
            MAPE.append(mape)
            std = np.std(abs(tp_py[i] - tp_ty[i]))
            STD.append(std)
            info = 'Oxygen sensor {}:   mae: {:.4f},  std: {:.4f}, mape: {:.4f}'.format(i, mae, std, mape)
            text_file.write(info + "\n")
            print('mae: ', mae, 'std: ', std, 'mape: ', mape)
        info = 'Temperature:   MAE: {:.4f},  STD: {:.4f}, MAPE: {:.4f}'.format(np.mean(MAE), np.mean(STD), np.mean(MAPE))
        text_file.write(info + "\n")

    #save predicted data
    test_time = test_Temp_time[47+6:]
    print(len(test_time), len(tp_ty[0]))

    for i in range(len(tp_py)):
        df = pd.DataFrame({'date': test_time,
                    'value': tp_py[i]})
        df.to_csv(root+'/prediction/' + ts_stem +str(sensor_idx[i])+'_LSTM_prediction.csv', index=False)
    ytest_lstm = tp_ty
    ypred_lstm = tp_py

In [ ]:
sel = 0
plt.plot(ytest_CNN[sel], 'k.', label = 'test')
plt.plot(ypred_CNN[sel], 'r.', label = 'CNN')
plt.plot(ypred_spatial[sel], 'g.', label = 'spatial')
plt.plot(ypred_lstm[sel], 'b.', label = 'LSTM')
plt.legend(loc = 'upper left',fontsize='xx-small')
plt.ylim([-1,1])
#plt.xlim([250,750])

In [ ]:
for i in range(len(ytest_CNN)):
    mae_CNN = mean_absolute_error(ytest_CNN[i], ypred_CNN[i])
    mae_lstm = mean_absolute_error(ytest_CNN[i], ypred_lstm[i])
    mae_spatial = mean_absolute_error(ytest_CNN[i], ypred_spatial[i])
    print(i, mae_CNN, mae_lstm, mae_spatial)

# Model

Model for SPATIAL and data cleaning

https://github.com/IBM/spatial-lstm/blob/a18fb9fda2d9c8e5311cf1f46e3173f82a21ad24/Model/model.py#L163

In [ ]:
# Data normalization:
# using log function and skip the masked data with -1
# 0 values are replaced by 1e-5 in order to avoid nan value in log
def data_normalize(Dat):
    new_dat  = []
    for d in Dat:
        min, max = np.min(sorted(list(set(d)))[1]), np.max(d)
        a = max -min
        temp = []
        for val in d:
            if val == -1: temp.append(val)
            else:
                norm =np.log(val) if val!=0 else 1e-5
                temp.append(norm ) #temp.append((val - min)/a )
        new_dat.append(temp)
    return 

In [ ]:
# interpolate the missing values with mask value
def interpolate(data, mask):
    temp = [list(dd) for dd in data]
    d = []
    for i in range(len(temp)):
        for j in range(len(temp[i])):
            if temp[i][j] == mask: temp[i][j]= float("NaN")
        df = pd.Series(temp[i]).interpolate(method='linear')
        d.append(df.tolist())
    return d

In [ ]:
#Continous data split for masked and unmasked data:
def split_train(Int_dat, Norm_dat,T1,T2,T3,Stride, start, end,data_name):
    length  = len(Int_dat[0])
    s = int(length*start)
    e = int(length*end)
    Train = [ N[:s] + N[e:] for N in Norm_dat ]
    Test  = [ M[s:e] for M in Int_dat ]
    print('Training Data Length: ', len(Train),'X',len(Train[0]))
    print('Test Data Length: ', len(Test), 'X',len(Test[0]))
    print('Testing percentage: ',len(Test[0])/(len(Test[0])+len(Train[0]))*100,'%' )
    print('Total data size: ', len(Int_dat), 'X', len(Int_dat[0]))
    np.savetxt(data_name + '_train.txt', np.exp(np.array(Train)))
    np.savetxt(data_name + '_test.txt', np.exp(np.array(Test)))
    train_x, train_y = data_split(Train, T1, T2, T3,Stride)
    test_x, test_y = data_split(Test, T1, T2, T3,Stride)
    return train_x, train_y, test_x, test_y

# split data for Multivairate time series (matrix of sensors)
# the data is feed to Bidirectional LSTM model
def data_split(dat, train_hour, test_hour, predict_position,  stride):
    #train_hour: training data length
    #test_hour: testing data length
    #predict_position: gap between train_hour and test_hour
    x, y = [], []
    period = train_hour + predict_position + test_hour
    i = 0
    while i + period <= len(dat[0]):
        x_temp = []
        y_temp = []
        for j in range(len(dat)):
            x_temp.append(dat[j][i:i + train_hour])
            y_temp.append(dat[j][i+ train_hour+ predict_position:i+ train_hour+ predict_position +test_hour])
        x.append(x_temp)
        y.append(y_temp)
        i += stride
    return np.array(x), np.array(y)

In [18]:
df.shape

(3, 2)

In [ ]:
#---------------------------Model-------------------------------------
# Bidirectional LSTM model
def stacked_LSTM(X, Y):
    time_step = X.shape[1]
    input_dim = X.shape[2] #qtd colunas 
    out = Y.shape[2]
    #Bidirectional LSTM
    start = time.time()
    model = Sequential()
    model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim))) #camada de entrada
    model.add(Bidirectional(LSTM(32,activation='elu', input_shape=(time_step, input_dim),return_sequences=True))) # camada escondida
    #model.add(Bidirectional(LSTM(64, activation='elu', input_shape=(time_step, input_dim), return_sequences=True)))
    #model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim)))
    model.add(Dense(out)) #camada saida
    model.compile(loss='mean_absolute_error', optimizer='adam')
    hist = model.fit(X, Y,epochs=100, validation_split=.2,
              verbose=0, batch_size=10)
    model.summary()
    end = time.time()
    print("Total compile time: --------", end - start, 's')
    return model, hist

In [ ]:
def SP_Learner(data, train_time, predict_time, predict_position,Stride, start, end, data_name):
    print('########################Start##################################')
    norm_dat = data_normalize(data) #normalized data (used for training)
    norm_int_dat = interpolate(norm_dat,-1) #normalized interpolated data (used for prediction)
    #-----------------------------------plot-------------------------------------------
    f1 = sns.heatmap(norm_dat)
    f1.set_title(data_name + ' Masked Data')
    plt.figure()
    plt.show()
    f2 = sns.heatmap(norm_int_dat)
    f2.set_title(data_name + ' Interpolated Data')
    plt.figure()
    plt.show()
    # -----------------------------------end plot-------------------------------------------
    # split the data set
    train_x, train_y, test_x, test_y = split_train(norm_int_dat, norm_dat, train_time, predict_time,predict_position, Stride,start, end, data_name)
    print('Train data size(batch, row, column)','Train X: ', train_x.shape,' ,Train Y: ',train_y.shape)
    print('test data size(batch, row, column)','Test X: ',test_x.shape,' ,Test Y: ',test_y.shape)
    # model training
    model, hist = stacked_LSTM(train_x,train_y)
    pred_y = model.predict(test_x, verbose=1)
    error, std = multi_heatmap(test_y, pred_y,predict_time,data_name)
    py = flattern(pred_y)
    ty = flattern(test_y)
    plt.figure()
    for j in range(len(ty)):
        plt.scatter(range(len(ty[j])),[ty[j][i]-py[j][i] for i in range(len(ty[j]))])
    plt.title(data_name + ' Test Errors')
    print('MAE: ', np.mean(error), 'STD: ',np.mean(std))
    print('########################End##################################')
    return py, ty, error, std, 